<a href="https://colab.research.google.com/github/murilopdr/murilo-76-tp546/blob/main/trabalho_pratico/FreezerGuard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FreezerGuard — tomada inteligente para proteção de freezers

**Trabalho final — Internet das Coisas (Inatel)**

**Problema.** Uma queda de energia, uma porta mal fechada ou um compressor parado podem estragar todo o conteúdo de um freezer, e o dono só descobre quando já é tarde. Para pequenos comércios (açougues, padarias, farmácias) o prejuízo é direto.

**Solução.** Uma tomada inteligente instalada entre a parede e o freezer. Ela mede **tensão** e **potência**, lê uma **sonda de temperatura** interna, um **sensor de porta** e um **sensor de temperatura ambiente**, envia tudo por **MQTT** e publica **alarmes** quando algo sai do normal. Pelo relé, o usuário também pode **ligar e desligar** o freezer remotamente.

**Neste notebook**, o freezer é simulado em Python, a tomada se comunica com a biblioteca `paho-mqtt` e a visualização e os comandos são feitos pelo **MQTTX**.

### Configuração

##### Instalação

In [1]:
!pip install -q "paho-mqtt>=2.0"

##### Comunicação MQTT

In [2]:
BROKER = "broker.emqx.io"              # broker público e gratuito
PORTA = 1883
TOPICO = "inatel/murilo/freezerguard"  # prefixo único: o broker é público
INTERVALO_S = 2                        # segundos entre leituras (cada leitura = 1 minuto simulado)

##### Simulação do freezer

In [3]:
TEMP_INICIAL = -18.0                   # temperatura interna inicial (°C)
TERMOSTATO_LIGA = -17.0                # compressor liga acima disso (°C)
TERMOSTATO_DESLIGA = -19.0             # compressor desliga abaixo disso (°C)
RESFRIAMENTO = 0.5                     # frio produzido pelo compressor (°C por minuto)
K_PORTA_FECHADA = 0.0035               # troca de calor com o ambiente, porta fechada (por minuto)
K_PORTA_ABERTA = 0.025                 # troca de calor com o ambiente, porta aberta (por minuto)

# A temperatura do ambiente é sorteada uma vez e mantida constante ao longo da simulação
import random
TEMP_AMB_MEDIA = 25.0                  # temperatura ambiente: média da gaussiana (°C)
TEMP_AMB_DESVIO = 2.0                  # temperatura ambiente: desvio-padrão (°C)
TEMP_AMBIENTE = random.gauss(TEMP_AMB_MEDIA, TEMP_AMB_DESVIO)
print(f"Temperatura ambiente sorteada: {TEMP_AMBIENTE:.1f} °C")

Temperatura ambiente sorteada: 28.0 °C


##### Alarmes

In [4]:
LIMITE_TEMP = -12.0                    # temperatura alta acima disso (°C)
TENSAO_MINIMA = 80.0                   # falta de energia abaixo disso (V)
PORTA_TEMPO_MAX_MIN = 3                # porta aberta por mais tempo que isso gera aviso (minutos)

# Gravidade de cada alarme (vai no payload MQTT e define a cor na saída)
SEVERIDADE = {
    "FALTA_DE_ENERGIA": "critico",
    "TEMPERATURA_ALTA": "critico",
    "PORTA_ABERTA": "aviso",
}

### Simulação do freezer

(descrever)

In [5]:
estado = {
    "minuto": 0,
    "temperatura": TEMP_INICIAL,  # temperatura interna real (°C)
    "compressor": False,
    "energia": True,              # há energia na rede?
    "porta_aberta": False,
    "rele": True,                 # relé da tomada (comandado remotamente)
}


def avancar_freezer():
    """Avança a física do freezer em 1 minuto."""
    estado["minuto"] += 1
    alimentado = estado["energia"] and estado["rele"]

    # Termostato do próprio freezer
    if not alimentado:
        estado["compressor"] = False
    elif estado["temperatura"] > TERMOSTATO_LIGA:
        estado["compressor"] = True
    elif estado["temperatura"] < TERMOSTATO_DESLIGA:
        estado["compressor"] = False

    # Calor que entra (lei de resfriamento de Newton) e frio do compressor
    k = K_PORTA_ABERTA if estado["porta_aberta"] else K_PORTA_FECHADA
    estado["temperatura"] += k * (TEMP_AMBIENTE - estado["temperatura"])
    if estado["compressor"]:
        estado["temperatura"] -= RESFRIAMENTO

### Lógica da tomada inteligente

In [6]:
def ler_sensores():
    """Leituras da tomada: o que os sensores medem (com ruído)."""
    alimentado = estado["energia"] and estado["rele"]
    if estado["compressor"]:
        potencia = random.gauss(120, 3)  # compressor funcionando
    elif alimentado:
        potencia = 2.0                   # só a eletrônica do freezer
    else:
        potencia = 0.0
    return {
        "minuto": estado["minuto"],
        "temperatura_c": round(estado["temperatura"] + random.gauss(0, 0.05), 2), # pequeno ruído de medição
        "temp_ambiente_c": round(TEMP_AMBIENTE + random.gauss(0, 0.05), 1),       # pequeno ruído de medição
        "tensao_v": round(random.gauss(127, 1), 1) if estado["energia"] else 0.0,
        "potencia_w": round(potencia, 1),
        "compressor": "ligado" if potencia > 50 else "desligado",  # inferido pela potência
        "porta": "aberta" if estado["porta_aberta"] else "fechada",
        "rele": "ligado" if estado["rele"] else "desligado",
    }

alarmes_ativos = set()
minutos_porta_aberta = 0  # minutos seguidos com a porta aberta

def verificar_alarmes(leitura):
    """Devolve a lista de alarmes que mudaram de estado nesta leitura."""
    global minutos_porta_aberta
    if leitura["porta"] == "aberta":
        minutos_porta_aberta += 1             # cada leitura = 1 minuto
        duracao_porta = minutos_porta_aberta
    else:
        duracao_porta = minutos_porta_aberta  # quanto tempo ficou aberta antes de fechar
        minutos_porta_aberta = 0

    condicoes = {
        "FALTA_DE_ENERGIA": leitura["tensao_v"] < TENSAO_MINIMA,
        "TEMPERATURA_ALTA": leitura["temperatura_c"] > LIMITE_TEMP,
        "PORTA_ABERTA": minutos_porta_aberta >= PORTA_TEMPO_MAX_MIN,
    }
    mudancas = []
    for nome, ativo in condicoes.items():
        if ativo == (nome in alarmes_ativos):
            continue  # nada mudou
        if ativo:
            alarmes_ativos.add(nome)
        else:
            alarmes_ativos.discard(nome)
        alarme = {"alarme": nome,
                  "estado": "ativo" if ativo else "encerrado",
                  "severidade": SEVERIDADE[nome],
                  "minuto": leitura["minuto"],
                  "temperatura_c": leitura["temperatura_c"]}
        if nome == "PORTA_ABERTA":
            alarme["minutos_porta_aberta"] = duracao_porta
        mudancas.append(alarme)
    return mudancas

### Comunicação MQTT

| Tópico | Sentido | QoS | Conteúdo |
|---|---|---|---|
| `.../telemetria` | tomada → nuvem | 0 | leituras a cada 2 s (JSON) |
| `.../alarme` | tomada → nuvem | 1 | alarme ativado ou encerrado (JSON) |
| `.../status` | tomada → nuvem | 1, *retained* | `online` / `offline` |
| `.../cmd` | nuvem → tomada | 1 | comandos em texto puro |

- **QoS:** a telemetria usa QoS 0 porque uma leitura perdida é substituída pela próxima, 2 s depois. Os alarmes usam QoS 1 (entrega garantida) porque não podem se perder.
- **Last Will:** registrado na conexão. Se a tomada cair sem se despedir, o próprio broker publica `offline` no lugar dela.
- **Retained:** o broker guarda o último status, então quem se conecta depois já vê se a tomada está online.

**Comandos.** `ligar` e `desligar` são comandos reais do produto. Os outros só existem na simulação, para provocar os eventos que, no mundo real, aconteceriam fisicamente.

In [22]:
import json
import paho.mqtt.client as mqtt

# comando -> (variável do estado, novo valor)
COMANDOS = {
    "ligar":         ("rele", True),           # produto: liga o freezer pelo relé
    "desligar":      ("rele", False),          # produto: desliga o freezer pelo relé
    "abrir_porta":   ("porta_aberta", True),   # simulação
    "fechar_porta":  ("porta_aberta", False),  # simulação
    "falta_energia": ("energia", False),       # simulação
    "volta_energia": ("energia", True),        # simulação
}

LINHA_AVISO = "~" * 115
NEGRITO_AVISO, RESET_AVISO = "\033[1m", "\033[0m"

def imprimir_aviso(texto):
    print(f"{NEGRITO_AVISO}{LINHA_AVISO}")
    print(texto)
    print(f"{LINHA_AVISO}{RESET_AVISO}")

def on_connect(client, userdata, flags, reason_code, properties):
    imprimir_aviso(f"Conectado ao broker {BROKER} ({reason_code})")
    client.subscribe(f"{TOPICO}/cmd", qos=1)
    client.publish(f"{TOPICO}/status", "online", qos=1, retain=True)

def on_message(client, userdata, msg):
    comando = msg.payload.decode().strip().lower()
    if comando in COMANDOS:
        variavel, valor = COMANDOS[comando]
        estado[variavel] = valor
        imprimir_aviso(f"Comando recebido: {comando}")
    else:
        imprimir_aviso(f"Comando desconhecido: {comando!r}. Opções: {', '.join(COMANDOS)}")

def conectar():
    """Cria o cliente MQTT, registra o Last Will e conecta ao broker."""
    client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)
    client.on_connect = on_connect
    client.on_message = on_message
    client.will_set(f"{TOPICO}/status", "offline", qos=1, retain=True)
    client.connect(BROKER, PORTA)
    client.loop_start()  # trata a rede em segundo plano
    return client

### Execução

A cada 2 s: o freezer avança 1 minuto, a tomada lê os sensores, publica a telemetria e publica os alarmes que mudaram.

Para parar, clique no botão ■ da célula: a tomada publica `offline` e desconecta. Para continuar de onde parou, execute esta célula de novo. Para recomeçar do zero, execute de novo a partir da seção 3.

In [23]:
import time

LARGURA = 115  # largura das faixas de alarme (igual à de uma linha de leitura)
VERMELHO, LARANJA, VERDE = "\033[91m", "\033[38;5;208m", "\033[92m"
NEGRITO, RESET = "\033[1m", "\033[0m"
ROTULO_CURTO = {"FALTA_DE_ENERGIA": "energia", "TEMPERATURA_ALTA": "temperatura", "PORTA_ABERTA": "porta"}


def imprimir_alarme(alarme):
    """Faixa vermelha (crítico) ou laranja (aviso) quando ativo; linha verde quando encerrado."""
    ativo = alarme["estado"] == "ativo"
    detalhes = f"min {alarme['minuto']}, {alarme['temperatura_c']:.2f} °C"
    if "minutos_porta_aberta" in alarme:
        verbo = "aberta há" if ativo else "ficou aberta por"
        detalhes += f", porta {verbo} {alarme['minutos_porta_aberta']} min"
    if ativo:
        if alarme["severidade"] == "critico":
            cor, rotulo = VERMELHO, "[!] ALARME CRÍTICO"
        else:
            cor, rotulo = LARANJA, "[!] AVISO"
        print(f"{cor}{NEGRITO}{'=' * LARGURA}")
        print(f"{rotulo}: {alarme['alarme']} ({detalhes})")
        print(f"{'=' * LARGURA}{RESET}")
    else:
        print(f"{VERDE}{'-' * LARGURA}")
        print(f"[ok] Alarme encerrado: {alarme['alarme']} ({detalhes})")
        print(f"{'-' * LARGURA}{RESET}")


def resumo_alarmes():
    """Alarmes ativos agora, na cor da gravidade (vermelho = crítico, laranja = aviso)."""
    itens = []
    for nome in SEVERIDADE:  # ordem fixa: a mesma da configuração
        if nome in alarmes_ativos:
            cor = VERMELHO if SEVERIDADE[nome] == "critico" else LARANJA
            itens.append(f"{cor}{ROTULO_CURTO[nome]}{RESET}")
    return "alarmes: " + (", ".join(itens) if itens else "-")

In [24]:
client = conectar()
alarmes_ativos.clear()  # como num reinício do dispositivo: reavalia todos os alarmes do zero

try:
    while True:
        avancar_freezer()
        leitura = ler_sensores()
        mudancas = verificar_alarmes(leitura)  # UMA única chamada por leitura
        client.publish(f"{TOPICO}/telemetria", json.dumps(leitura), qos=0)
        print(f"min {leitura['minuto']:4d} | {leitura['temperatura_c']:7.2f} °C | "
              f"{leitura['potencia_w']:6.1f} W | {leitura['tensao_v']:6.1f} V | "
              f"porta {leitura['porta']:7s} | relé {leitura['rele']:9s} | {resumo_alarmes()}")

        for alarme in mudancas:
            client.publish(f"{TOPICO}/alarme", json.dumps(alarme), qos=1)
            imprimir_alarme(alarme)

        time.sleep(INTERVALO_S)
except KeyboardInterrupt:
    print("Parado pelo usuário.")
finally:
    client.publish(f"{TOPICO}/status", "offline", qos=1, retain=True).wait_for_publish(timeout=2)
    client.disconnect()
    client.loop_stop()

min  451 |   25.13 °C |    0.0 W |    0.0 V | porta fechada | relé ligado    | alarmes: energia, temperatura
[!] ALARME CRÍTICO: FALTA_DE_ENERGIA (min 451, 25.13 °C)
[!] ALARME CRÍTICO: TEMPERATURA_ALTA (min 451, 25.13 °C)
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Conectado ao broker broker.emqx.io (Success)
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
min  452 |   25.14 °C |    0.0 W |    0.0 V | porta fechada | relé ligado    | alarmes: energia, temperatura
min  453 |   25.12 °C |    0.0 W |    0.0 V | porta fechada | relé ligado    | alarmes: energia, temperatura
min  454 |   25.31 °C |    0.0 W |    0.0 V | porta fechada | relé ligado    | alarmes: energia, temperatura
min  455 |   25.16 °C |    0.0 W |    0.0 V | porta fechada | relé ligado    | alarmes: energia, temperatura
min  456 |   25.22 °C |    0.0 W |    0.0 V | porta fechada | re